In [0]:
-- ============================================================
-- PARAMETERS
-- ============================================================

CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';


In [0]:
-- ============================================================
-- L1: PLAN → PAYER MAP
-- Ensures no plan duplication explosion
-- Grain: 1 row per PLAN_ID
-- ============================================================

CREATE OR REPLACE TEMP VIEW payer_plan_map AS
SELECT
  KH_PLAN_ID AS PLAN_ID,
  MAX(PAYER_ID)      AS PAYER_ID,
  MAX(PAYER_NAME)    AS PAYER_NAME,
  MAX(PARENT_ID)     AS PARENT_ID,
  MAX(PARENT_NAME)   AS PARENT_NAME,
  MAX(INSURANCE_GROUP) AS INSURANCE_GROUP
FROM com_edp_prd.com_raw.kom_plans
WHERE PAYER_ID IS NOT NULL
GROUP BY KH_PLAN_ID;

---------Validation - Must return zero rows

SELECT PLAN_ID, COUNT(*)
FROM payer_plan_map
GROUP BY PLAN_ID
HAVING COUNT(*) > 1;


In [0]:
-- ============================================================
-- L2: TOTAL LIVES PATIENT-PLAN BASE
-- Grain: 1 row per PATIENT_ID + PLAN_ID
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives_patient_plan AS

WITH medical AS (
  SELECT DISTINCT
    PATIENT_ID,
    KH_PLAN_ID AS PLAN_ID
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    AND KH_PLAN_ID IS NOT NULL
),

pharmacy AS (
  SELECT DISTINCT
    PATIENT_ID,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS PLAN_ID
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
    AND COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) IS NOT NULL
)

SELECT DISTINCT *
FROM (
  SELECT * FROM medical
  UNION
  SELECT * FROM pharmacy
);


In [0]:
-- ============================================================
-- L2b: TOTAL LIVES PATIENT-PAYER
-- ============================================================

CREATE OR REPLACE TEMP VIEW total_lives_patient_payer AS
SELECT DISTINCT
  t.PATIENT_ID,
  p.PAYER_ID,
  p.PAYER_NAME,
  p.PARENT_ID,
  p.PARENT_NAME,
  p.INSURANCE_GROUP
FROM total_lives_patient_plan t
JOIN payer_plan_map p
  ON t.PLAN_ID = p.PLAN_ID;


In [0]:
CREATE OR REPLACE TEMP VIEW MPSII_TREATMENT_TABLE AS

WITH base AS (

    -- 1️⃣ Medical NDC
    SELECT DISTINCT
        m.PATIENT_ID,
        COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
        m.BILLING_NPI                              AS HCO_NPI,
        m.NDC11                                    AS CODE,
        m.MEDICAL_EVENT_ID                         AS EVENT_ID,
        m.SERVICE_DATE                             AS FILL_DATE,
        m.KH_PLAN_ID                               AS PLAN_ID
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
    WHERE m.NDC11 IN ('54092070001','540920700')

    UNION

    -- 2️⃣ Paid Pharmacy NDC
    SELECT DISTINCT
        ph.PATIENT_ID,
        ph.PRESCRIBER_NPI                           AS HCP_NPI,
        ph.PHARMACY_NPI                             AS HCO_NPI,
        ph.NDC11                                    AS CODE,
        ph.PHARMACY_EVENT_ID                        AS EVENT_ID,
        ph.FILL_DATE                                AS FILL_DATE,
        COALESCE(ph.PRIMARY_KH_PLAN_ID, ph.SECONDARY_KH_PLAN_ID)
                                                     AS PLAN_ID
    FROM com_edp_prd.com_raw.kom_pharmacy_events ph
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON ph.PATIENT_ID = pat.PATIENT_ID
    WHERE ph.NDC11 IN ('54092070001','540920700')
      AND ph.TRANSACTION_RESULT = 'PAID'

    UNION

    -- 3️⃣ Medical Procedure Codes
    SELECT DISTINCT
        m.PATIENT_ID,
        COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
        m.BILLING_NPI                              AS HCO_NPI,
        m.PROCEDURE_CODE                           AS CODE,
        m.MEDICAL_EVENT_ID                         AS EVENT_ID,
        m.SERVICE_DATE                             AS FILL_DATE,
        m.KH_PLAN_ID                               AS PLAN_ID
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master pat
        ON m.PATIENT_ID = pat.PATIENT_ID
    WHERE m.PROCEDURE_CODE IN (
        'J1743','99601','99602','96365','96366',
        'S9357','S9379',
        '38206','38230','38232','38240',
        '38241','38242','38243','38250'
    )

)

SELECT *
FROM base
WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';
  -- AND PLAN_ID IS NOT NULL;


---------- Validation -- 7 PATIENTS DROPPING BECAUSE THEY DON'T HAVE PLAN ID MAPPED TO ANY OF THEIR ELAPRASE TREATMENT CLAIMS

SELECT COUNT(DISTINCT PATIENT_ID)
FROM MPSII_TREATMENT_TABLE;


In [0]:
SELECT * FROM MPSII_TREATMENT_TABLE;

In [0]:
--------ARE WE MAPPING LATEST NON NULL PLAN ONLY?
CREATE OR REPLACE TEMP VIEW patient_latest_event AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY PATIENT_ID
               ORDER BY FILL_DATE DESC
           ) AS rn
    FROM MPSII_TREATMENT_TABLE
    where PLAN_ID IS NOT NULL
)
WHERE rn = 1;


In [0]:
CREATE OR REPLACE TEMP VIEW hcp_geo_map AS

WITH hcp_base AS (

    -- ============================================================
    -- Step 1: Clean HCP + Extract ZIP5
    -- ============================================================

    SELECT DISTINCT
        TRIM(CAST(NPI AS STRING)) AS HCP_ID,
        PRIMARY_SPECIALTY         AS SPECIALTY,
        TRY_CAST(
            SUBSTR(
                REGEXP_REPLACE(COALESCE(PROVIDER_ZIP,''),'[^0-9]',''),
                1, 5
            ) AS BIGINT
        ) AS ZIP5
    FROM com_edp_prd.com_raw.kom_providers
    WHERE NPI IS NOT NULL
      AND PROVIDER_TYPE = 'INDIVIDUAL'
),

hcp_with_geo AS (

    -- ============================================================
    -- Step 2: ZIP → Territory Mapping
    -- ============================================================

    SELECT DISTINCT
        h.HCP_ID,
        h.SPECIALTY,

        COALESCE(TRY_CAST(z.territory_id AS BIGINT), -2) AS territory_id,
        COALESCE(z.territory_name, 'UNKNOWN')            AS territory_name,
        COALESCE(TRY_CAST(z.region_id AS BIGINT), -2)    AS region_id,
        COALESCE(z.region_name, 'UNKNOWN')               AS region_name

    FROM hcp_base h
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON h.ZIP5 = z.zipcode
)

-- ============================================================
-- Step 3: Enforce 1 HCP → 1 Territory
-- Deterministic selection using lowest territory_id
-- ============================================================

SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY HCP_ID
               ORDER BY territory_id
           ) AS rn
    FROM hcp_with_geo
)
WHERE rn = 1;



------- Validation

SELECT * FROM hcp_geo_map;

In [0]:
CREATE OR REPLACE TEMP VIEW payer360_patient_spine AS
SELECT DISTINCT
    e.EVENT_ID                        AS CLAIM_ID,
    e.PATIENT_ID,

    -- Primary HCP
    e.HCP_NPI                         AS PRIMARY_HCP,
    ref.hcp_name                      AS PRIMARY_HCP_NAME,
    ref.hcp_primary_specialty         AS PRIMARY_HCP_SPECIALTY,
    -- ref.hcp_primary_email             AS PRIMARY_HCP_EMAIL,

    -- Primary HCO
    ref.HCO_NPI                         AS PRIMARY_HCO,
    ref.HCO_NAME                        AS PRIMARY_HCO_NAME,
    ref.hco_state                       AS PRIMARY_HCO_STATE,

    -- Territory mapping via HCP
    g.territory_id                    AS PRIMARY_TERRITORY_ID,
    g.territory_name                  AS PRIMARY_TERRITORY_NAME,
    g.region_id,
    g.region_name,



    -- Latest Plan
    e.PLAN_ID,

    -- Payer Info (direct from kom_plans)
    p.PAYER_ID,
    p.PAYER_NAME,
    p.PARENT_ID,
    p.PARENT_NAME,
    p.INSURANCE_GROUP                 AS INSURANCE_SEGMENT

FROM patient_latest_event e

LEFT JOIN com_edp_prd.com_raw.kom_plans p
    ON e.PLAN_ID = p.KH_PLAN_ID

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file_0219 ref
    ON e.HCP_NPI = ref.HCP_NPI
    
LEFT JOIN hcp_geo_map g
  ON e.HCP_NPI = g.HCP_ID;

SELECT * FROM payer360_patient_spine;


In [0]:
SELECT COUNT(DISTINCT PATIENT_ID) FROM payer360_patient_spine WHERE PRIMARY_HCP IS NULL;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_0219;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master;

In [0]:
---============================================================
/* ELAPRASE PATIENT METRICS
PURPOSE:
Count Elaprase patients by payer + territory
GRAIN:
PAYER_ID + TERRITORY_ID
*/
---============================================================

CREATE OR REPLACE TEMP VIEW agg_elaprase_patients AS
SELECT
    PAYER_ID,
    PAYER_NAME,
    PRIMARY_TERRITORY_ID AS territory_id,
    PRIMARY_TERRITORY_NAME AS territory_name,
    region_id,
    region_name,
    PARENT_ID,
    PARENT_NAME,

    COUNT(DISTINCT PATIENT_ID) AS TOTAL_ELAPRASE_PATIENTS

FROM payer360_patient_spine
GROUP BY
    PAYER_ID, PAYER_NAME,
    PRIMARY_TERRITORY_ID,
    PRIMARY_TERRITORY_NAME,
    region_id, region_name,
    PARENT_ID, PARENT_NAME;


-------- Validation

SELECT * FROM agg_elaprase_patients;

In [0]:
---============================================================
/* INSURANCE SEGMENT SPLIT
*/
---============================================================

CREATE OR REPLACE TEMP VIEW agg_insurance_split AS
SELECT
    PAYER_ID,
    PRIMARY_TERRITORY_ID AS territory_id,

    COUNT(DISTINCT CASE WHEN UPPER(INSURANCE_SEGMENT)='MEDICARE'
         THEN PATIENT_ID END) AS MEDICARE_PATIENTS,

    COUNT(DISTINCT CASE WHEN UPPER(INSURANCE_SEGMENT)='MEDICAID'
         THEN PATIENT_ID END) AS MEDICAID_PATIENTS,

    COUNT(DISTINCT CASE WHEN UPPER(INSURANCE_SEGMENT)='COMMERCIAL'
         THEN PATIENT_ID END) AS COMMERCIAL_PATIENTS,

    COUNT(DISTINCT CASE WHEN INSURANCE_SEGMENT NOT IN
         ('MEDICARE','MEDICAID','COMMERCIAL')
         THEN PATIENT_ID END) AS OTHER_PATIENTS

FROM payer360_patient_spine
GROUP BY PAYER_ID, PRIMARY_TERRITORY_ID;

-------- Validation

SELECT * FROM agg_insurance_split;


In [0]:
---============================================================
/* NEW PATIENTS (R1M / R3M)
PURPOSE:
Based on FIRST Elaprase fill date
*/
---============================================================



-------- Validation

SELECT * FROM agg_insurance_split;
